# Solving combinatorial problems with QAOA

This notebook demonstrates the process of solving a combinatorial problem using the QUBO notation and the QAOA VQA.

In this example we will be tackling the traveling salesman problem (TSP) which aims at finding the lowest cost cycle in a graph while visiting each node once.

In [1]:
import networkx # used here to generate the graph
import numpy as np
import numpy.typing as npt
import random

First we will be generating the graph.
For this example we will be taking a 3-node graph with positive weights.

In [7]:
nbr_nodes = 3
G = networkx.fast_gnp_random_graph(nbr_nodes,1) # generates n nodes with every node connected to each other
cost_list = []
for (u,v,w) in G.edges(data=True):
    w['weight'] = random.randint(3,10)
    cost_list.append(([u,v],w['weight']))
    cost_list.append([[v,u], w['weight'] + random.randint(-2,10)])
print(G)
print()
print(cost_list)

Graph with 3 nodes and 3 edges

[([0, 1], 5), [[1, 0], 15], ([0, 2], 7), [[2, 0], 5], ([1, 2], 4), [[2, 1], 9]]


Now that we have our graph with random weights we need to get the adjacency matrix of this graph.

This is a matrix that represent all of the weights between each nodes.

In [ ]:
def generate_adjacency_matrix(list_of_cost : list[tuple[list[int], int]], nb_cities: int):
    """Example of list of cost = [[[0,1],23], [[1,2], 52], [[0,2], 12]] """
    w_ij = np.zeros((nb_cities, nb_cities), dtype=int)
    for i in list_of_cost:
        cost = i[1]
        w_ij[i[0][0], i[0][1]] = cost
    return w_ij
adj_matrix = generate_adjacency_matrix(cost_list,3)
print(adj_matrix) 

Now that we have our matrix we will be generating our QUBO expression.
A QUBO (quadratic unconstrained binary optimization) is a quadratic polynomial expression of binary variables, it is used to represent different combinatorial problems.

The goal is to find the set of binary values that minimize the expression.

For example if we have the expression : $2*x_0 - 3*x_1\\$
Then the minimum value will be -3 with $x_0 = 0$ and $x_1 = 1$.


For each path between two nodes we need to define a QUBO monomial so we have 2*edges.

In [ ]:
from mpqp.execution.vqa.qubo import Qubo

def generate_qubo_expression(adj_matrix : npt.NDArray[np.int64]) -> Qubo :
    length = len(adj_matrix)
    result = Qubo('0', flag=True)
    for i in range(length):
        for j in range(length):
            if adj_matrix[i][j] == 0:
                continue
            current = adj_matrix[i][j] * Qubo(f'x{i},{j}')
            if result.value == '0':
                result = current
            else:
                result += current 
    return result if not isinstance(result,int) else 0*Qubo("x") # if the adjacency matrix is empty

expr = generate_qubo_expression(adj_matrix)

expr.pprint()
initial_coeffs = expr.get_coeffs([])
print(initial_coeffs)

Now that we have our Qubo expression we need to implement the constraints of this problem, constraints here 

The first one is the fact that we have to go through each cities exactly once, in this case we express it as xor operators between every path that end with the same node.

For example if the path $x_{0,1}$ is chosen then the path $x_{2,1}$ cannot be chosen because they both lead to the same node.

The second constraint is the inverse because it is not possible to start from the same city twice so once again the path $x_{0,1}$ excludes the path $x_{0,2}$.

The way penalties works is when minimizing the loss function (our QUBO) we can enforce some behavior so that the minimum of the function respects the rules of the problem.
Here because we need to go through each cities once so it translates to xor operations with a negative coefficient hence if the xor is true it will decrease the value of the loss function.

Note : The coefficient of penalties can decide if the minimization finds the correct answer, if the penalties are too high then the minimization won't find the difference between solutions as long as they respect the constraints. However, if the penalties are not high enough the minimum of the loss function can ignore the penalties.

In [ ]:
def add_constraints(expr : Qubo, nbr_nodes: int) -> Qubo:
    for i in range(nbr_nodes):
        if i != 0:
            current = Qubo(f'x{i},0')
        else:
            current = Qubo(f'x{i},1')
        for j in range(1,nbr_nodes):
            if i == j or (i == 0 and j == 1):
                continue
            path = Qubo(f'x{i},{j}')
            print(f'{current.value} ^ {path.value}')
            expr += -20* (current ^ path)
    for i in range(nbr_nodes):
        if i != 0:
            current = Qubo(f'x0,{i}')
        else:
            current = Qubo(f'x1,{i}')
        for j in range(1,nbr_nodes):
            if i == j or (i == 0 and j == 1):
                continue
            path = Qubo(f'x{j},{i}')
            print(f'{current.value} ^ {path.value}')
            expr += -20* (current ^ path)
    return expr

expr = add_constraints(expr, nbr_nodes)

In [ ]:
from mpqp.execution.vqa.qaoa import MixerType, qaoa_solver

def test_results(res : str, coeffs : list[tuple[int, list[str]]]):
    current_answer = 0
    other = 0
    for i in range(len(res)):
        if res[i] == '0':
            other += coeffs[i][0]
        else:
            current_answer += coeffs[i][0]
    return current_answer <= other

result = qaoa_solver(expr, 3,MixerType.MIXER_X, 'Nelder-Mead')

print(f"The path chose : {result} is the best path : {test_results(result,initial_coeffs)}")